In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
import os
import warnings
import matplotlib.pyplot as plt

In [ ]:
###  ['e', 'd2m', 'sde', 'sp', 't2m', 'tp', 'stl1', 'stl2', 'stl3', 'stl4', 'swvl1', 'swvl2', 'swvl3', 'swvl4']
###  ['ro', 'v10', 'ssr', 'ssrd', 'str', 'strd']
###  ['e', 'd2m', 'sde', 'sp', 't2m', 'tp', 'ro', 'v10', 'ssr', 'ssrd', 'str', 'strd']
# var: e, rh(d2m), sde, sp, t2m, tp, ro, v10, ssr, ssrd, str, strd
# stmi: stl1, stl2, stl3, stl4 swvl1, swvl2, swvl3, swvl4

In [ ]:
# 计算空间相对湿度
import xarray as xr
import numpy as np

# 定义你已经提供的函数

def calculate_saturation_vapor_pressure(temperature_C):
    """
    计算给定温度下的饱和水蒸气压（单位：hPa）
    使用 Tetens 公式
    :param temperature_C: 温度，单位：摄氏度
    :return: 水蒸气饱和压力，单位：hPa
    """
    return 6.11 * 10**(7.5 * temperature_C / (temperature_C + 237.3))

def calculate_relative_humidity(temperature_C, dew_point_C):
    """
    计算相对湿度
    :param temperature_C: 当前空气温度，单位：摄氏度
    :param dew_point_C: 露点温度，单位：摄氏度
    :return: 相对湿度，单位：百分比
    """
    # 计算当前温度下的水蒸气压力 E(T)
    E_T = calculate_saturation_vapor_pressure(temperature_C)
    
    # 计算露点温度下的饱和水蒸气压力 E(T_d)
    E_Td = calculate_saturation_vapor_pressure(dew_point_C)
    
    # 计算相对湿度 RH
    RH = (E_Td / E_T) * 100
    return RH

# 读取 NetCDF 文件
file_path = "D:/Permafrost/EN_MASK/var/nc/d2m_sub.nc"  # 替换为你文件的路径
file_path1 = "D:/Permafrost/EN_MASK/var/nc/t2m_sub.nc"  # 替换为你文件的路径
dataset = xr.open_dataset(file_path)
dataset1 = xr.open_dataset(file_path1)

# 获取温度和露点温度的数据（假设文件中这两个变量名分别为 'temperature' 和 'dew_point'）
dew_point_data = dataset['d2m'] -273.15  # 假设变量名为 'temperature'
temperature_data = dataset1['t2m'] -273.15  # 假设变量名为 'dew_point'

# 批量计算相对湿度
relative_humidity_data = calculate_relative_humidity(temperature_data, dew_point_data)

# 将计算出的相对湿度数据存储回 NetCDF 文件
relative_humidity_da = xr.DataArray(relative_humidity_data, 
                                    dims=['valid_time', 'latitude', 'longitude'],  # 根据实际数据维度设置
                                    coords={'valid_time': dataset.coords['valid_time'],
                                            'latitude': dataset.coords['latitude'],
                                            'longitude': dataset.coords['longitude']})
relative_humidity_da.name = 'rh'

# 保存为新的 NetCDF 文件
output_file_path = "D:/Permafrost/EN_MASK/var/nc/rh_sub.nc"
relative_humidity_da.to_netcdf(output_file_path)

# 关闭数据集
dataset.close()
dataset1.close()

print(f"相对湿度计算完成，结果已保存至 {output_file_path}")

相对湿度计算完成，结果已保存至 D:/Permafrost/EN_MASK/var/nc/rh_sub.nc
